# 30-day mortality, Bayesian logistic regression (Synthea)

**This notebook is not executed.** The flat Synthea CSV it was written against is no
longer on disk, so the cells below have never been run in this form. Read it as a
reference implementation, not as a results transcript.

## What happened to the outputs

An earlier run of this model existed with outputs attached. Two things make those
numbers unsafe to carry forward:

1. **The design matrix was collinear.** The original formula was
   `0 + age_per_decade + C(gender) + C(chief_complaint)`. The `0 +` prefix does not
   remove the intercept when the contrasts are `Treatment`, which is patsy's default.
   You get `Intercept` *plus* all the dummy columns: 10 columns of rank 9. PyMC cannot
   identify the posterior when the likelihood is flat in that direction, and the
   divergences and `r_hat` values from that run cannot be trusted.
2. The source data is gone, so none of it can be checked or reproduced.

The design matrix below is the fixed version. To get it, the `0 +` prefix is dropped
and the intercept is removed from the matrix instead, leaving a full-rank
1 + 1 + 1 + 7 = 10 columns that the model samples cleanly.

## To run it

You need the flat Synthea 5k CSV with one row per patient. The three columns used
here are `age`, `gender`, `chief_complaint` (Synthea calls this `reason`, on the
encounter), plus `died_30days` derived from `DEATH` against the encounter date.
The `george-essam-ai/synthea-healthcare-dataset` repo on HuggingFace is the raw
relational export (parquet, one file per table); it does *not* contain the flat CSV
and cannot be used without rebuilding the table first.

Set `DATA` below to your file and run top to bottom.

In [ ]:
import numpy as np
import pandas as pd

import arviz as az
import pymc as pm
import pytensor.tensor as pt
import pytensor
from patsy import dmatrix
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold

pytensor.config.cxx = ""          # no C compiler in this env; use the python linker
DATA = "synthea_5k_flat.csv"      # <- point this at your file
LABEL = "died_30days"
SEED = 42

df = pd.read_csv(DATA)
print(df.shape)

In [ ]:
AGE_MIN, AGE_MAX = 18, 90

# per decade, so a coefficient of 0.3 means something legible on this scale
work = df.copy()
work["age_per_decade"] = (work["age"] - AGE_MIN) / 10.0

FORMULA = "age_per_decade + C(gender) + C(chief_complaint)"

# no "0 +" prefix: Treatment contrasts keep their intercept by design,
# so instead of fighting patsy we drop the column afterwards. 10 columns,
# rank 10, nothing to identify.
X = dmatrix(FORMULA, work, return_type="dataframe")
X = X.drop(columns=["Intercept"])
print(X.shape, "| rank check:", np.linalg.matrix_rank(X.values), "of", X.shape[1])

y = work[LABEL].astype(int).values
print(f"{y.sum()} deaths of {len(y)} patients ({y.mean():.1%})")

In [ ]:
COORDS = {"feature": X.columns.tolist()}

with pm.Model(coords=COORDS) as model:
    alpha = pm.Normal("alpha", mu=0.0, sigma=1.5)
    beta = pm.Normal("beta", mu=0.0, sigma=1.5, dims="feature")
    p = pm.math.sigmoid(alpha + pt.dot(X.values, beta))
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y)
    idata = pm.sample(
        draws=2000, tune=2000, chains=4, target_accept=0.9,
        random_seed=SEED, progressbar=False,
    )

az.summary(idata, var_names=["alpha", "beta"])

In [ ]:
summary = az.summary(idata, var_names=["alpha", "beta"])
print("max r_hat:", summary["r_hat"].max())
print("min ess_bulk:", int(summary["ess_bulk"].min()))
print("divergences:", int(idata.sample_stats["diverging"].sum()))

post = idata.posterior["beta"]
odds_ratio = az.summary(az.convert_to_dataset(post)).loc[:, ["mean", "hdi_3%", "hdi_97%"]]
print(np.exp(odds_ratio))

In [ ]:
post_b = idata.posterior["beta"].values.reshape(-1, X.shape[1])
post_a = idata.posterior["alpha"].values.reshape(-1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
auc, apc, bri = [], [], []

for tr, te in skf.split(X.values, y):
    with pm.Model(coords=COORDS):
        a = pm.Normal("alpha", mu=0.0, sigma=1.5)
        b = pm.Normal("beta", mu=0.0, sigma=1.5, dims="feature")
        p = pm.math.sigmoid(a + pt.dot(X.values[tr], b))
        pm.Bernoulli("y_obs", p=p, observed=y[tr])
        fold = pm.sample(draws=1000, tune=1000, chains=4, target_accept=0.9,
                       random_seed=SEED, progressbar=False)

    pb = fold.posterior["beta"].values.reshape(-1, X.shape[1])
    pa = fold.posterior["alpha"].values.reshape(-1)
    p_test = 1 / (1 + np.exp(-(pa[:, None] + pb @ X.values[te].T))).mean(axis=0)

    auc.append(roc_auc_score(y[te], p_test))
    apc.append(average_precision_score(y[te], p_test))
    bri.append(brier_score_loss(y[te], p_test))

print(f"AUROC  {np.mean(auc):.3f} +/- {np.std(auc):.3f}")
print(f"AUPRC  {np.mean(apc):.3f}   (base rate {y.mean():.3f})")
print(f"Brier  {np.mean(bri):.4f}")